# Event-Driven Agentic AI

**Fiodar Sazanavets** — Senior AI Engineer and Microsoft MVP  
@FSazanavets | https://scientificprogrammer.net

---

This notebook covers the following topics:

- [ ] Foundations of event-driven agentic systems — terminology, event sources, and core use cases
- [ ] Architectural patterns — event buses, pub/sub topologies, state consistency, pitfalls, and performance tuning
- [ ] Frameworks and tooling — messaging/streaming platforms, orchestration engines, agent stacks, and cloud event routers
- [ ] Designing real-time agent workflows — event-native agent design, inter-agent protocols, and human-in-the-loop
- [ ] Making event-driven agents robust — failure taxonomy, retry strategies, idempotency, and state machines
- [ ] Observability, monitoring, and security — traces, metrics, logs, identity/access, and data privacy

---

## Part 1 — Foundations of Event-Driven Agentic Systems

### What is Event-Driven Architecture?

**Event-driven architecture** is a software design pattern where components communicate by producing and consuming events rather than calling each other directly.

Key properties:
- **Loose coupling** — producers and consumers are independent
- **Asynchronous** — components do not wait for each other
- **Scalable** — consumers can be scaled independently
- **Resilient** — failures in one component do not cascade

### Core Terminology

| Term | Definition |
|------|------------|
| **Event** | A record of something that happened — immutable fact about the past |
| **Command** | An instruction to do something — directed, expects a result |
| **Fact** | A statement about the current state of the world |
| **Stream** | An ordered, append-only sequence of events |
| **Saga** | A long-running workflow composed of a sequence of local transactions |

Events are **named in the past tense** (e.g., `OrderPlaced`, `PaymentProcessed`).  
Commands are **named as imperatives** (e.g., `PlaceOrder`, `ProcessPayment`).

### Push vs. Poll

There are two fundamental patterns for receiving data from a source:

**Push** — the source sends data to the consumer when it is available.
- Lower latency
- Consumer must be ready to receive
- Examples: webhooks, server-sent events, WebSockets

**Poll** — the consumer periodically asks the source for new data.
- Simpler to implement
- Introduces latency proportional to the polling interval
- Wastes resources when there is nothing new
- Examples: REST polling, batch jobs

### Event Sources

#### Webhooks

An HTTP callback that delivers event notifications from a source system to a registered endpoint.

- **Push-based** — no polling required
- **Simple** — just an HTTP POST to your URL
- Requires the consumer endpoint to be publicly reachable
- Must handle retries and duplicate delivery

#### Message Queues

A durable buffer that decouples producers from consumers.

- Each message is consumed by **one consumer** (competing consumers pattern)
- Messages are deleted after successful processing
- Provides **at-least-once** delivery guarantees
- Examples: Azure Queue Storage, AWS SQS, RabbitMQ

#### Pub/Sub

A messaging pattern where producers publish to topics and multiple consumers subscribe independently.

- Each subscriber receives **its own copy** of the message
- Enables fan-out to multiple consumers
- Examples: Azure Service Bus topics, Google Cloud Pub/Sub, NATS

#### Streaming

An append-only, ordered log of events that can be replayed.

- Consumers track their own **offset** (position in the stream)
- Events are retained for a configurable period
- Supports replay, time-travel, and multiple independent consumers
- Examples: Apache Kafka, Azure Event Hubs, Redis Streams

### IoT and MQTT

**MQTT** (Message Queuing Telemetry Transport) is a lightweight publish/subscribe protocol designed for constrained devices and unreliable networks.

- **QoS 0** — at most once (fire and forget)
- **QoS 1** — at least once (acknowledged delivery)
- **QoS 2** — exactly once (four-part handshake)
- Used extensively in IoT, industrial automation, and edge computing
- Devices publish sensor readings; agents subscribe to act on them

### Change Data Capture (CDC)

CDC turns database row-level changes into a stream of events.

- Reads the database **transaction log** (e.g., binlog, WAL)
- Every `INSERT`, `UPDATE`, and `DELETE` becomes an event
- Zero impact on the application — no code changes required
- Enables real-time integration between legacy systems and event-driven agents
- Tools: Debezium, AWS DMS, Azure Data Factory

### Agentic Use Cases for Event-Driven Systems

#### Automated Monitoring
- Agent subscribes to a metrics stream
- Detects anomalies and threshold breaches in real time
- Takes corrective action autonomously or escalates

#### Incident Triage
- Alert event triggers the agent
- Agent gathers context: logs, metrics, recent deployments
- Classifies severity and routes to the right team

#### Workflow Approval
- Business event (e.g., large purchase) triggers approval workflow
- Agent collects required information and presents it to approvers
- Tracks SLA and escalates if approval is not received in time

#### RAG Refresh
- Document change event triggers the agent
- Agent re-chunks, re-embeds, and upserts into the vector store
- Keeps the retrieval index fresh without manual intervention

#### Business Process Monitoring
- Agent monitors the event stream for process deviations
- Detects stuck orders, missed SLAs, or fraud patterns
- Initiates compensating actions automatically

#### IT Ops Automation
- Infrastructure events (CPU spike, disk full) trigger the agent
- Agent diagnoses root cause and applies a fix
- Escalates to on-call engineer only when needed

---

## Part 2 — Architectural Patterns for Event-Driven Agents

### Event Buses and Pub/Sub Topologies

#### Central Event Bus

All producers publish to a single bus; all consumers subscribe from it.

**Pros:**
- Single place for schema governance and observability
- Easy to add new consumers

**Cons:**
- Single point of failure
- Can become a bottleneck at high throughput
- Coupling through shared infrastructure

#### Federated Event Bus

Multiple domain-specific buses connected via bridges or replication.

**Pros:**
- Better isolation between domains
- Scales independently per domain
- Failure in one domain does not affect others

**Cons:**
- More complex to operate
- Cross-domain routing adds latency

### Kafka vs. NATS

| Dimension | Apache Kafka | NATS |
|-----------|-------------|------|
| **Model** | Append-only partitioned log | Pub/sub with optional JetStream persistence |
| **Throughput** | Millions of messages/sec | Millions of messages/sec |
| **Latency** | Low (ms) | Very low (sub-ms) |
| **Replay** | Yes — configurable retention | JetStream only |
| **Ordering** | Per partition | Per subject |
| **Best for** | High-volume streams, audit log, CDC | Fast coordination, IoT, microservice calls |
| **Cloud managed** | Azure Event Hubs, Confluent | Synadia Cloud |

### Topic Design

A well-designed topic naming convention is the foundation of a maintainable event-driven system.

**Recommended pattern:** `<domain>.<entity>.<event-type>`

Examples:
- `orders.order.placed`
- `payments.payment.processed`
- `inventory.stock.updated`

**Partitioning strategy:**
- Partition by entity ID to preserve ordering for a given entity
- Avoid hot partitions — distribute load evenly
- Consider time-based partitioning for time-series data

**Schema governance:**
- Register schemas in a schema registry (e.g., Confluent Schema Registry, AWS Glue)
- Enforce backward/forward compatibility rules
- Version schemas explicitly

### State Consistency

#### Sagas

A saga is a sequence of local transactions coordinated by events. If any step fails, compensating transactions undo previous steps.

**Choreography saga** — each service publishes events that trigger the next service. No central coordinator.

**Orchestration saga** — a central orchestrator issues commands to each participant and handles failures.

#### The Outbox Pattern

Ensures that a database write and an event publication happen atomically — even if the message broker is temporarily unavailable.

1. Write business state **and** an outbox record in the **same database transaction**
2. A background worker polls the outbox table and publishes pending events
3. Mark outbox records as published after successful delivery

This guarantees **at-least-once** delivery with no data loss.

#### Idempotency Keys

Because events may be delivered more than once, consumers must be idempotent — processing the same event twice produces the same result as processing it once.

- Attach a unique `event-id` or `correlation-id` to every event
- Consumer records processed IDs in a deduplication store
- On re-delivery, skip processing if the ID is already recorded

#### Exactly-Once vs. Effectively-Once

| Guarantee | Description |
|-----------|-------------|
| **At most once** | Events may be lost; never duplicated |
| **At least once** | Events are never lost; may be duplicated |
| **Exactly once** | True transactional guarantee — expensive, requires broker support |
| **Effectively once** | At-least-once + idempotent consumer — practical alternative |

#### Optimistic Concurrency

Prevent conflicting updates by attaching a version token (ETag / rowversion) to each event. The consumer checks the version before committing. If it has changed, the operation is retried.

### Architectural Pitfalls

#### Event Storms
An event triggers multiple downstream events that each trigger more events, causing exponential fan-out.
- **Fix:** Rate limiting, circuit breakers, and careful topology design.

#### Hot Partitions
Too many events routed to the same partition, creating a bottleneck.
- **Fix:** Choose a partition key with high cardinality; add synthetic randomness if needed.

#### Duplicate Deliveries
The same event is processed more than once due to retries or at-least-once semantics.
- **Fix:** Idempotency keys and deduplication stores.

#### Poison Messages
A malformed or unprocessable message blocks the queue or partition.
- **Fix:** Dead-letter queue (DLQ) — move failed messages out of the main path after N retries.

#### Head-of-Line Blocking
A slow or stuck message prevents all subsequent messages from being processed.
- **Fix:** Multiple consumer threads, DLQ after max retries, or priority queues.

#### Fan-out Bottlenecks
A single event must be delivered to thousands of consumers, overwhelming the broker.
- **Fix:** Consumer groups, hierarchical topics, or async fan-out with buffering.

#### Cold Starts
Serverless/container-based agents suffer latency when woken by an event after being idle.
- **Fix:** Keep-alive pings, minimum instance counts, or warm pool strategies.

### Latency, Throughput, and Backpressure Tuning

#### Batching
Group multiple events into a single network round-trip to increase throughput.
- Increases throughput at the cost of slightly higher latency
- Configure `linger.ms` and `batch.size` in Kafka producers

#### Windowing
Process events in time-bounded or count-bounded windows.
- **Tumbling window** — fixed non-overlapping intervals
- **Sliding window** — overlapping intervals
- **Session window** — gaps in activity delimit boundaries

#### Consumer Groups
Multiple consumer instances share the load of a topic by each reading from a different partition subset.

#### Rate Limiting
Protect downstream systems by capping the number of events processed per unit of time.

#### Circuit Breakers
If a downstream dependency is failing, stop sending it requests and fail fast instead of queuing up retries.

---

## Part 3 — Frameworks and Tooling for Event-Driven Agents

### Messaging and Streaming Platform Decision Guide

#### High-Volume Streams: Kafka / Azure Event Hubs

Use when you need **durable, replayable, high-throughput** event streams.

| | Apache Kafka | Azure Event Hubs |
|-|-------------|------------------|
| **Protocol** | Kafka native | Kafka-compatible + AMQP |
| **Retention** | Configurable (days/bytes) | Up to 90 days (Premium/Dedicated) |
| **Partitions** | Per topic | Per Event Hub |
| **Managed** | Self-hosted or Confluent Cloud | Fully managed by Azure |
| **Best for** | On-prem/multi-cloud | Azure-first workloads |

#### Fast Coordination: NATS / Azure Service Bus

Use when you need **low-latency request/reply, work queues, or topic subscriptions**.

| | NATS | Azure Service Bus |
|-|------|-----------------|
| **Model** | Pub/sub + request/reply | Queues + topics/subscriptions |
| **Latency** | Sub-millisecond | Low (ms) |
| **Persistence** | JetStream (optional) | Always durable |
| **DLQ** | JetStream | Built-in |
| **Best for** | Microservice coordination, IoT | Enterprise messaging, workflow |

#### Lightweight Queues: Redis Streams / Azure Queue Storage

Use when you need **simple, lightweight task queues** without complex topology.

| | Redis Streams | Azure Queue Storage |
|-|--------------|--------------------|
| **Model** | Append-only log with consumer groups | Simple FIFO queue |
| **Persistence** | In-memory + optional AOF/RDB | Durable by default |
| **Max message size** | No hard limit | 64 KB |
| **Best for** | Real-time pipelines with existing Redis | Simple, cheap task queuing |

### Orchestration Engines

Long-running agent workflows require durable state management so they can survive restarts, failures, and wait for external events.

#### Azure Durable Functions

An extension of Azure Functions that adds **stateful, durable workflow orchestration**.

- **Orchestrator functions** — coordinate the workflow; automatically checkpointed
- **Activity functions** — individual steps; called by the orchestrator
- **Entity functions** — stateful actors
- Patterns: function chaining, fan-out/fan-in, async HTTP, monitoring, human interaction
- Language support: C#, JavaScript, Python, Java, PowerShell

#### AWS Step Functions

A visual, fully managed state machine service for orchestrating AWS services and custom code.

- Workflows defined in **Amazon States Language** (JSON)
- **Standard workflows** — exactly-once execution, audit history, up to 1 year
- **Express workflows** — at-least-once, high throughput, up to 5 minutes
- Built-in integrations with 200+ AWS services
- Visual canvas for designing and debugging workflows

### Agent Stacks

#### Semantic Kernel

Microsoft's open-source SDK for integrating LLMs into applications.

- **Kernel** — the central orchestrator; manages plugins, memory, and planners
- **Plugins** — collections of functions (skills) the agent can call
- **Planners** — generate a plan of action from a goal
- Native integration with Azure OpenAI, OpenAI, and Hugging Face
- Strong .NET support; Python and Java SDKs also available
- Built-in connectors for vector stores and memory

#### LangChain / LangGraph

Python/JavaScript framework for building LLM-powered applications.

- **LangChain** — chains, agents, tools, memory, and retrieval primitives
- **LangGraph** — graph-based orchestration for stateful multi-agent workflows
  - Nodes are agent steps; edges are conditional transitions
  - Supports cycles (unlike DAGs) — essential for agentic loops
  - Built-in persistence and human-in-the-loop checkpointing

#### AutoGen

Microsoft Research framework for building multi-agent conversational systems.

- Agents communicate by exchanging messages in a conversation
- **AssistantAgent** — LLM-powered; generates responses and calls tools
- **UserProxyAgent** — executes code and relays results
- Supports group chat, nested agents, and custom conversation patterns
- AutoGen Studio provides a no-code interface for rapid prototyping

### Cloud Event Routers

Cloud event routers decouple event producers from consumers at the infrastructure level, providing filtering, transformation, and routing.

#### Azure Event Grid

- Fully managed pub/sub event routing service
- **Event sources:** Azure services (Blob Storage, Resource Manager, IoT Hub), custom topics, partners
- **Event handlers:** Azure Functions, Logic Apps, Service Bus, Event Hubs, webhooks
- Filter events by event type, subject prefix/suffix, or any custom attribute
- Supports CloudEvents 1.0 schema
- Built-in retry with exponential backoff; dead-letter destination

#### AWS EventBridge

- Serverless event bus connecting AWS services, SaaS apps, and custom sources
- **Event buses:** default (AWS services), custom, partner
- Rule-based routing with JSON pattern matching
- Schema registry with auto-discovery and code binding generation
- Pipes: point-to-point integrations with filtering and enrichment
- Global endpoints for cross-region failover

#### GCP Eventarc

- Routes events from Google Cloud services and custom sources to Cloud Run, Cloud Functions, and GKE
- Native CloudEvents support
- Triggers based on Audit Logs, Pub/Sub messages, or direct events
- Integrates with Workflows for multi-step orchestration

---

## Part 4 — Designing Real-Time Agent Workflows

### Event-Native Agent Design

An **event-native agent** is designed from the ground up to be triggered by events, react asynchronously, and emit events as output — rather than being bolted onto an event bus as an afterthought.

#### Steps Inside an Agent

1. **Trigger** — agent wakes up in response to an event
2. **Guards** — should we even run? (deduplication, relevance check, authorization)
3. **Tool selection** — choose the right actions based on the context
4. **Memory attachment** — load relevant context from semantic and episodic memory

#### Memory Types

| Type | Purpose |
|------|---------|
| **Semantic memory** | Fuels reasoning; search of historic vector stores; past decisions inform current ones |
| **Episodic memory** | Session-specific facts; recent steps and outcomes; keeps the context window tight |

### Inter-Agent Protocols

In multi-agent systems, agents must communicate using well-defined protocols to collaborate effectively.

#### Topics

- Define where events go
- Allow agents to subscribe to events
- Multiple agents can listen to the same events
- Further per-subscription filtering on: subject, file extension, any custom event data

#### Roles

- Define which agents do what
- Required for multi-agent interactions
- Separation of responsibility
- Limiting per-agent context

#### Message Schemas

- Define the shape of the messages
- Pre-agreed communication format
- Schemas matter because they:
  - Prevent hallucinated fields
  - Allow validation at ingestion
  - Enable versioning and evolution
  - Support replay and audit

#### Shared Scratchpad

A shared memory space where agents record collaborative reasoning artifacts:
- What agents remember together
- Intermediate reasoning artifacts
- Partial plans
- Decisions already made
- Facts discovered so far

#### Delegation Contracts

How work is handed off between agents. The contract defines:
- Task scope (what is delegated)
- Input schema
- Expected output
- Timeout / SLA
- Escalation path
- Authority boundaries

### Human-in-the-Loop and SLAs

**Human-in-the-loop** is the pattern where agents pause and wait for a sign-off or additional information from a real person before proceeding.

When to involve a human:
- Decision exceeds the agent's authority boundary
- Confidence is below a threshold
- Action is irreversible (e.g., deleting data, issuing a refund)
- Regulatory or compliance requirement

#### SLA-Driven Human-in-the-Loop

To prevent the approval process from blocking the workflow indefinitely:

1. **Time-limit the decision** — e.g., "refund decision in 15 minutes"
2. **Escalate if time runs out** — PagerDuty, email, Teams notification, etc.
3. **Fallback** — automated action such as a partial refund if no response
4. **Rate cap** — approval requests do not overwhelm reviewers

---

## Part 5 — Making Event-Driven Agents Robust

### Failure Taxonomy

Understanding the type of failure is the first step to handling it correctly.

#### Transient Errors

- Temporary issues — network blips, brief throttling, cold caches
- **Resolution:** retries with exponential backoff

#### Systemic Errors

- Widespread issues — region outage, partition hot spots, dependency outage
- **Resolution:** load shedding, circuit breaker, escalation

#### Logical Errors

- Bugs or bad assumptions — schema mismatches, null dereferences, malformed payloads
- **Resolution:** isolated by dead-lettering and quarantine; fixed by patches and updates

### Retries Done Right

> Retries shouldn't overwhelm the system and cause more problems than they solve.

#### Retry Techniques

| Technique | Description |
|-----------|-------------|
| **Exponential backoff** | Retry in 1s, 2s, 4s, 8s, etc. — progressively longer waits |
| **Jitter** | Randomize the wait time to avoid synchronized thundering herds |
| **Max attempts** | Stop when several retries fail — don't retry forever |
| **Poison queue** | Failed messages do not block the route — moved aside |
| **Dead-letter queue** | Isolate the message for further examination and manual intervention |

### Idempotency Strategies

#### Idempotency Keys

- A unique identifier attached to each request or event
- An **effect log** stores outcomes keyed by this identifier
- A duplicate key does not cause a duplicate effect
- Prevents double charges, double sends, and double writes

#### Semantic Idempotency

- The operation is inherently safe to repeat (e.g., setting a flag to `true`)
- Requires no extra storage
- Not possible for operations with side effects like payments

#### State Machines

Model the workflow as a finite state machine to prevent illegal state transitions:

- Progress only moves forward — no going back to a previous state
- Skips the step if it is already completed
- Excellent for human-in-the-loop workflows
- Good for implementing timeouts

#### The Outbox Pattern (Revisited)

- Write business state and the outbox record in the **same transaction**
- Background worker publishes the event
- Publisher is idempotent — safe to retry if the process crashes after publishing but before marking as done

---

## Part 6 — Observability, Monitoring, and Security

### Traces, Metrics, and Logs

The three pillars of observability provide complementary views of system behavior.

#### How Traces Work

A **trace** represents the end-to-end journey of a single request or event through the system:

- **Root span** — denotes the operation boundaries (e.g., event received → action completed)
- **Child spans** — cover sub-operations (LLM call, tool invocation, database write)
- Linked together via a **correlation ID** that propagates through all services
- Tools: OpenTelemetry, Azure Application Insights, AWS X-Ray, Jaeger

#### The Importance of Metrics

Metrics are high-volume, low-cost numerical measurements:

- Simple values: counts, durations, error rates, queue depths
- Can be visualized in dashboards (Grafana, Azure Monitor)
- Can be linked to alerting rules
- Key agent metrics: events processed/sec, LLM token usage, tool call latency, retry rate

#### The Purpose of Logs

Logs provide detailed, narrative descriptions of individual events:

- Detailed description of individual events
- Can form traces when correlated by trace ID
- Can use templates and formatters for structured logging
- Come with different severity levels: TRACE, DEBUG, INFO, WARN, ERROR, FATAL
- Lower severity levels can be disabled in production to reduce cost

### Identity and Access

> Guard against unauthorized access. Prove that you are who you are claiming to be. Prove that you are allowed to access.

#### Authentication vs. Authorization

| Concept | Definition |
|---------|------------|
| **Authentication** | Proof that you are who you are claiming to be |
| **Authorization** | Proof that you are permitted to perform the action |

#### Managed Identity

- An identity given to an Azure service — no need to store secrets in code or config
- Permissions configured at the service level in Azure RBAC
- Eliminates credential rotation burden
- Supported by Azure Service Bus, Event Hubs, Event Grid, and more

#### Mutual TLS with X.509 Certificates

- Both parties present certificates to authenticate each other
- Used in device-to-device communication and MQTT
- Device certificate is validated against a trusted CA
- Prevents unauthorized devices from publishing events

#### Role-Based Access Control (RBAC)

- Defines who is allowed to do what on which resource
- Combine multiple roles together to grant least-privilege access
- Fine-grained access control at topic/queue level
- Roles can be assigned or revoked dynamically

#### Private Endpoint

- The service endpoint is invisible and inaccessible from the public internet
- Accessible only from within the private virtual network
- Can be accessed via encrypted VPN or ExpressRoute tunnel

### Data Security and Privacy

#### Enforcing Privacy and Data Security

| Practice | Description |
|----------|-------------|
| **PII redacting** | No sensitive information shown in logs, traces, or events |
| **Allowlist approach** | Explicitly define what is permitted — deny everything else |
| **Payload encryption** | Do not transfer raw sensitive data — encrypt at field or envelope level |
| **Key governance** | Store keys securely (Azure Key Vault, AWS Secrets Manager) and rotate them |
| **Token governance** | Prefer short-lived tokens; understand the risk profile of long-lived tokens |

### Policy and Guardrails

Guardrails constrain agent behavior to prevent unsafe, unauthorized, or undesirable actions.

#### Content Filters

> Limit the scope of the agent. Don't allow it to do what it's not supposed to do.

- Block requests that fall outside the agent's defined scope
- Filter inputs and outputs for harmful content
- Use Azure AI Content Safety, Amazon Bedrock Guardrails, or custom classifiers

#### Approval Gates

> Auto-approve below a certain threshold. Then get a sign-off from a human.

- Actions below a risk/cost threshold proceed automatically
- Actions above the threshold are paused for human review
- Threshold is configurable per use case and can adapt over time

---

## Further Reading

- [Enterprise Integration Patterns](https://www.enterpriseintegrationpatterns.com/) — Hohpe & Woolf — the definitive reference for messaging patterns
- [Apache Kafka Documentation](https://kafka.apache.org/documentation/) — official Kafka docs
- [NATS Documentation](https://docs.nats.io/) — official NATS docs
- [Azure Event Hubs Documentation](https://learn.microsoft.com/azure/event-hubs/) — Microsoft Learn
- [Azure Service Bus Documentation](https://learn.microsoft.com/azure/service-bus-messaging/) — Microsoft Learn
- [Azure Durable Functions](https://learn.microsoft.com/azure/azure-functions/durable/durable-functions-overview) — orchestration patterns and usage
- [AWS Step Functions Developer Guide](https://docs.aws.amazon.com/step-functions/latest/dg/welcome.html) — state machine orchestration
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/) — graph-based agent orchestration
- [Semantic Kernel Documentation](https://learn.microsoft.com/semantic-kernel/) — Microsoft's AI orchestration SDK
- [AutoGen Documentation](https://microsoft.github.io/autogen/) — multi-agent conversation framework
- [OpenTelemetry Documentation](https://opentelemetry.io/docs/) — vendor-neutral observability instrumentation
- [The Outbox Pattern](https://microservices.io/patterns/data/transactional-outbox.html) — microservices.io pattern reference
- [Saga Pattern](https://microservices.io/patterns/data/saga.html) — distributed transaction coordination